## Example 1: Documents: untyped JSON-like values with typed accessors

In [ ]:
import importlib.metadata

from smithy_core.documents import Document
from smithy_core.exceptions import ExpectationNotMetError

# Example 1: a Document wraps nested values; as_value() unwraps them, the typed accessors check the kind
assert importlib.metadata.version("smithy-core") == "0.9.0"
doc = Document({"name": Document("run-7"), "epochs": Document(3), "tags": Document([Document("gpu")])})
assert doc.as_value() == {"name": "run-7", "epochs": 3, "tags": ["gpu"]}, doc.as_value()
assert doc["epochs"].as_integer() == 3 and doc["tags"].as_list()[0].as_string() == "gpu"
try:
    doc["name"].as_integer()
    raise AssertionError("a string document read as an integer")
except ExpectationNotMetError as error:
    print("Example 1 OK:", type(error).__name__)

## Example 2: Shape IDs and URIs

In [ ]:
from smithy_core import URI
from smithy_core.shapes import ShapeID

# Example 2: Smithy shape identifiers and the URI type every client request is built on
shape = ShapeID("com.amazonaws.bedrockruntime#Converse")
assert shape.namespace == "com.amazonaws.bedrockruntime" and shape.name == "Converse", (shape.namespace, shape.name)
uri = URI(host="bedrock-runtime.us-east-1.amazonaws.com", path="/model/m/converse", query="a=1")
assert uri.build() == "https://bedrock-runtime.us-east-1.amazonaws.com/model/m/converse?a=1", uri.build()
assert uri.netloc == "bedrock-runtime.us-east-1.amazonaws.com"
print("Example 2 OK:", shape, uri.build())

## Example 3: Exponential retry backoff

In [ ]:
from smithy_core.retries import ExponentialBackoffJitterType, ExponentialRetryBackoffStrategy

# Example 3: no wait before the first try, then the delay doubles per retry up to the cap; with no jitter it is exact
strategy = ExponentialRetryBackoffStrategy(backoff_scale_value=0.1, max_backoff=1.0, jitter_type=ExponentialBackoffJitterType.NONE)
delays = [strategy.compute_next_backoff_delay(i) for i in range(7)]
assert delays == [0, 0.1, 0.2, 0.4, 0.8, 1.0, 1.0], delays
full = ExponentialRetryBackoffStrategy(jitter_type=ExponentialBackoffJitterType.FULL, random=lambda: 0.5)
assert 0 <= full.compute_next_backoff_delay(3) <= 0.025 * 2 ** 3
print("Example 3 OK:", delays)

## Example 4: Schemas and the prelude types

In [ ]:
from smithy_core import prelude
from smithy_core.schemas import Schema
from smithy_core.shapes import ShapeType

# Example 4: every Smithy shape has a Schema; the prelude defines the simple ones
assert prelude.STRING.shape_type == ShapeType.STRING and prelude.INTEGER.shape_type == ShapeType.INTEGER
assert prelude.DOCUMENT.shape_type == ShapeType.DOCUMENT
assert isinstance(prelude.BLOB, Schema) and str(prelude.STRING.id) == "smithy.api#String", prelude.STRING.id
print("Example 4 OK:", prelude.STRING.id, prelude.TIMESTAMP.shape_type)

## Example 5: Errors and retry information

In [ ]:
from smithy_core.exceptions import CallError, SmithyError

# Example 5: client errors carry whether and how they may be retried
error = CallError(message="throttled", fault="server", is_retry_safe=True, retry_after=1.5, is_throttling_error=True)
assert isinstance(error, SmithyError) and error.is_retry_safe and error.retry_after == 1.5 and error.is_throttling_error
other = CallError(message="bad input", fault="client", is_retry_safe=False)
assert not other.is_retry_safe and other.fault == "client"
print("Example 5 OK:", error, "|", other)